# CCF-opt line mask from recovery ``inj0`` L2 (fiber 2, RV=0)

Regenerate a **static rest-frame** ``<template_stem>.ccfopt.delta.mas`` from the
**same** RV=0 L2 product tree as ``rv_recovery_sim.ipynb`` (not a second OUT /
fiberflat / WAVE). Prefer running the rebuild cell inside ``rv_recovery_sim`` after
``inj0_rv0``; this notebook is the standalone path that loads that L2.

Uses :data:`~hispecdrp.utils.line_mask_utils.CCFOPT_SINGLE_LINE_STELLAR_RV_KW`
(line-by-line, ``±15`` km/s @ ``25`` m/s, ``ccf_input_velocity=0``):

1. Load blaze-corrected L2 from ``RECOVERY_OUT/inj0_rv0/`` (produced by
   ``rv_recovery_sim`` with ``poisson_noise=False``, shared Aidan+ZP₀ WAVE,
   shared fiberflat, barycorr skipped).
2. For **each** seed line, single-line line-by-line CCF → shift
   ``λ' = λ × (1 + RV/c)`` so that line's RV → 0; weight ∝ `|contrast|`.
3. Write ``hispecdrp/data/masks/<stem>.ccfopt.delta.mas`` (static for ±500 km/s;
   do **not** re-zero at −50 km/s).
4. Closed-loop full-mask stellar RV check at RV=0.

**Prerequisite:** run ``rv_recovery_sim.ipynb`` through the RV=0 cell first (or set
``L2_PATH`` to an existing inj0 ``*_spec.fits``).

> Set `MAX_SEED_LINES` only for a quick smoke test.


In [ ]:
import os
from pathlib import Path

import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# Same product tree as rv_recovery_sim (do not invent a second fiberflat/WAVE).
RECOVERY_OUT = "/Users/ckilpatrick/hispec_rv_recovery_sim"
OUT = str(Path(RECOVERY_OUT) / "ccfopt_mask_from_inj0")
os.makedirs(OUT, exist_ok=True)

# Prefer recovery's KOA cache so Stage-0/1 provenance matches inj0.
os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(RECOVERY_OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("RECOVERY_OUT:", RECOVERY_OUT)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Locate recovery ``inj0`` L2 (skip re-sim)

KOA sync is owned by ``rv_recovery_sim``; this notebook only needs the blaze L2.


In [ ]:
from pathlib import Path

SPECTROGRAPH = "BSPEC"
INJ0_DIR = Path(RECOVERY_OUT) / "inj0_rv0"
# Prefer blaze-corrected L2 without a stellar_rv suffix.
_candidates = sorted(INJ0_DIR.glob("HB.*_spec.fits"))
_candidates = [p for p in _candidates if "stellar_rv" not in p.name]
if not _candidates:
    raise FileNotFoundError(
        f"No inj0 L2 under {INJ0_DIR}. Run rv_recovery_sim.ipynb through the "
        "RV=0 cell first (poisson_noise=False path)."
    )
L2_PATH = _candidates[-1]
print(f"Using recovery inj0 L2: {L2_PATH}")


## 2. Template / fiber / mask paths


In [ ]:
from hispecdrp.utils.line_mask_utils import (
    CCFOPT_SINGLE_LINE_STELLAR_RV_KW,
    package_data_masks_dir,
)

TEMPLATE = "lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5"
TEMPLATE_STEM = Path(TEMPLATE).stem
FIBER = 2
FIBERS = [FIBER]

MASKS_DIR = package_data_masks_dir()
SEED_MASK = MASKS_DIR / f"{TEMPLATE_STEM}.delta.mas"
OUT_MASK = MASKS_DIR / f"{TEMPLATE_STEM}.ccfopt.delta.mas"

# None = use the full seed; set an int for a quick smoke test.
MAX_SEED_LINES = None

assert SEED_MASK.is_file(), f"missing seed mask: {SEED_MASK}"
print(f"template: {TEMPLATE}")
print(f"fiber={FIBER}")
print(f"L2_PATH: {L2_PATH}")
print(f"seed mask: {SEED_MASK}")
print(f"output mask: {OUT_MASK}")
print(f"CCFOPT_SINGLE_LINE_STELLAR_RV_KW: {CCFOPT_SINGLE_LINE_STELLAR_RV_KW}")
print(f"MAX_SEED_LINES={MAX_SEED_LINES!r}")


## 3. Load recovery inj0 L2

No re-simulation here — apples-to-apples with ``rv_recovery_sim`` means one shared L2.


In [ ]:
import numpy as np

from hispecdrp import datamodels

run_dir = OUT
os.makedirs(run_dir, exist_ok=True)
l2 = datamodels.open(str(L2_PATH))
wave = np.asarray(getattr(l2, f"wave_fib{FIBER}"), dtype=np.float64)
print(f"Loaded L2: {getattr(l2.meta, 'filename', L2_PATH)}")
print(
    f"wave_fib{FIBER} range: [{np.nanmin(wave):.4f}, {np.nanmax(wave):.4f}] µm"
)


## 4. Optional seed-mask subset


In [ ]:
from hispecdrp.utils.line_mask_utils import MaskLines

seed_ml = MaskLines.load(str(SEED_MASK))
n_seed_pos = int(np.sum(np.asarray(seed_ml.weight) > 0))
if MAX_SEED_LINES is not None:
    n_keep = int(MAX_SEED_LINES)
    lam = np.asarray(seed_ml.line_lambda_um, dtype=np.float64)
    w = np.asarray(seed_ml.weight, dtype=np.float64)
    ok = np.isfinite(lam) & (lam > 0) & np.isfinite(w) & (w > 0)
    idx = np.flatnonzero(ok)[:n_keep]
    tmp_seed = Path(OUT) / f"{TEMPLATE_STEM}.seed_subset.delta.mas"
    MaskLines.from_delta_centers_um(lam[idx], w[idx] / np.sum(w[idx])).save(
        output_path=str(tmp_seed)
    )
    mask_path = str(tmp_seed)
    print(f"Using subset seed ({n_keep} lines): {mask_path}")
else:
    mask_path = str(SEED_MASK)
    print(f"Using full seed mask ({n_seed_pos} lines)")


## 5. Single-line line-by-line CCF zeroing → ``*.ccfopt.delta.mas``

For each seed line, run **line-by-line** CCF with a **one-line** mask
(``CCFOPT_SINGLE_LINE_STELLAR_RV_KW`` — same as ``rv_recovery_sim``):

- ``λ' = λ_seed × (1 + RV/c)`` so that line's recovered RV → 0
- weight ∝ `|Gaussian contrast|` (renormalized to sum 1)

Then re-run full-mask stellar RV (``ccf_zb=0``, ``ccf_zb_from_barycorr=False``)
to confirm closed-loop bias ≈ 0. Mask stays static for ±500 km/s (no −50 re-zero).


In [ ]:
from hispecdrp.primitives.stellar_rv.stellar_rv_primitive import StellarRVPrimitive
from hispecdrp.utils.line_mask_utils import (
    CCFOPT_SINGLE_LINE_STELLAR_RV_KW as _CCFOPT_RV,
    MaskLines,
    build_ccfopt_line_mask_from_l2_single_line_ccf,
    read_mask_generation_meta,
)

tpl_path = Path(hispecdrp.__file__).resolve().parent / "data" / "templates" / TEMPLATE
print("Building static ccfopt mask via single-line line-by-line CCF ...")
path, entries = build_ccfopt_line_mask_from_l2_single_line_ccf(
    mask_path,
    l2,
    fiber=FIBER,
    template_path=str(tpl_path),
    output_path=str(OUT_MASK),
    masks_directory=str(MASKS_DIR),
    max_seed_lines=MAX_SEED_LINES,
)
meta = read_mask_generation_meta(path)
print(f"wrote {len(entries)} lines -> {path}")
print(
    f"n_seed={meta.get('n_seed_lines')}  n_drop={meta.get('n_dropped_lines')}  "
    f"weight_def={meta.get('mask_weight_definition')}"
)
print(
    f"seed single-line RV median={meta.get('seed_single_line_rv_median_m_s'):.3f} m/s  "
    f"MAD={meta.get('seed_single_line_rv_mad_m_s'):.3f} m/s"
)

ml = MaskLines.load(path)
w = np.asarray(ml.weight, dtype=np.float64)
print(f"weight sum = {float(np.sum(w)):.9f}  n_positive = {int(np.sum(w > 0))}")

print("\nClosed-loop: full ccfopt mask, line-by-line stellar RV + preview ...")
prim = StellarRVPrimitive(
    algorithm="ccf_line_mask",
    mask_path=str(path),
    do_fibers=list(FIBERS),
    ccf_input_velocity=0.0,
    window_line_by_line=bool(_CCFOPT_RV["window_line_by_line"]),
    window_size_um=float(_CCFOPT_RV["window_size_um"]),
    v_half_window=float(_CCFOPT_RV["v_half_window_m_s"]),
    v_step_coarse=float(_CCFOPT_RV["v_step_m_s"]),
    mask_auto_center_scout=bool(_CCFOPT_RV["mask_auto_center_scout"]),
    mask_ccf_pixel_interp_order=int(_CCFOPT_RV["mask_ccf_pixel_interp_order"]),
    ccf_zb=0.0,
    ccf_zb_from_barycorr=False,
    save_preview=True,
    show_preview=False,
    preview_write_png=False,
    save_result=True,
    output_dir=run_dir,
    suffix="stellar_rv_ccfopt_check",
)
l2_chk = prim.apply(l2)
avg = getattr(l2_chk, f"rvccfavg_fib{FIBER}", None)
tbl = getattr(l2_chk, f"rvccf_fib{FIBER}", None)
if avg is None or len(avg) == 0:
    raise RuntimeError(f"missing rvccfavg_fib{FIBER} after closed-loop check")
print(
    f"RVCCFAVG: {float(avg['RV'][0]):.6f} ± {float(avg['RV_ERR'][0]):.6f} m/s  "
    f"(n_chunks={len(tbl) if tbl is not None else 0})"
)
preview_html = l2_chk.get_output_path(
    output_dir=run_dir, suffix="stellar_rv_ccfopt_check"
).replace(".fits", "_preview.html")
print(f"preview: {preview_html}")

# Per-order mean of line RVs (should be ~0 after line-by-line zeroing).
if tbl is not None and "ORDER" in tbl.colnames and "RV" in tbl.colnames:
    orders = np.asarray(tbl["ORDER"], dtype=int)
    rvs = np.asarray(tbl["RV"], dtype=float)
    print("\nPer-order mean line RV (m/s):")
    for o in np.unique(orders[np.isfinite(rvs)]):
        m = (orders == o) & np.isfinite(rvs)
        print(
            f"  order {int(o):3d}: n={int(np.sum(m)):4d}  "
            f"mean={float(np.mean(rvs[m])):+8.3f}  "
            f"median={float(np.median(rvs[m])):+8.3f}"
        )
